# Devign E05 — Routing policy

## A. Title and description

**Experiment:** E05 routing policy. **Goal:** run exactly one direct-high, verify-high, or reproduced-baseline policy and create descriptive routing curves. **Inputs:** Devign/model assets, optional E01 artifacts, and optionally a prior run checkpoint. **Outputs:** model artifacts, resumable checkpoints, routing metrics, operating points, curves, and matched-recall CSV. **LLM:** full direct-high calls LLM for inspect only; verify-high calls it for inspect and high; reproduced baseline forces all samples through LLM. **Sessions:** one policy at a time; inference may span multiple chunked sessions.

**Important:** smoke-test results are development checks and must not be used in the paper.


## B. User configuration

Edit only this centralized cell before a run.


In [ ]:
REPOSITORY_URL = "https://github.com/khanhtran0111/VulGuardVN.git"
BRANCH = "camera-ready"
DATASET = "devign"

RUN_MODE = "smoke"       # "smoke", "full", or "dry-run"
SEED = 42
CONFIGURATION = "verify_high"

OUTPUT_ROOT = "/kaggle/working/revision_results"
SMOKE_OUTPUT_ROOT = "/kaggle/working/revision_smoke_results"

AUTO_DOWNLOAD_MODEL = False
RESUME = True
FORCE_RECLONE = False
RESTORE_CHECKPOINT = False
CHECKPOINT_INPUT = "/kaggle/input/vulguard-devign-e05-checkpoint"
TEST_CHUNK_SIZE = 250       # Set None to disable multi-session inference chunks.
TEST_CHUNK_INDEX = None      # None automatically selects the next unresolved chunk.

# Only used by experiments that read E01 artifacts.
REUSE_E01_RESULTS = True
E01_RESULTS_INPUT = "/kaggle/input/vulguard-devign-e01-results"

SESSION_BUDGET_HOURS = 11.5
MIN_REMAINING_MINUTES = 20
EXPERIMENT = "E05_routing_policy"
# CONFIGURATION: "direct_high", "verify_high", or "reproduced_baseline".
SOURCE_EXPERIMENT = "E01_multiseed"


## C. Kaggle environment checks

Checks working directory, Python, disk, NVIDIA driver, CUDA visibility, GPU name/memory, and UTC start time.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os, platform, shutil, subprocess

KAGGLE_WORKING = Path("/kaggle/working")
ON_KAGGLE = KAGGLE_WORKING.exists() and str(Path.cwd()).startswith("/kaggle")
START_TIME = datetime.now(timezone.utc)
print("Kaggle environment:", ON_KAGGLE)
if not ON_KAGGLE:
    print("WARNING: this notebook is intended for /kaggle/working; use dry-run outside Kaggle.")
print("Python:", platform.python_version())
disk_root = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()
usage = shutil.disk_usage(disk_root)
print("Disk GB:", {"total": round(usage.total/2**30, 2), "free": round(usage.free/2**30, 2)})
subprocess.run(["nvidia-smi"], check=False)
try:
    import torch
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory GB:", round(props.total_memory/2**30, 2))
    elif True:
        print("WARNING: this experiment needs GPU for UniXcoder and/or LLM execution.")
except Exception as exc:
    print("WARNING: PyTorch/GPU check failed:", exc)
print("Start time UTC:", START_TIME.isoformat())


## D. Prepare repository

Clone `camera-ready` when absent; otherwise fetch, checkout, and pull the target branch.


In [ ]:
from pathlib import Path
import shutil, subprocess

REPO_DIR = Path("/kaggle/working/VulGuardVN")
if not Path("/kaggle/working").exists() and Path.cwd().name == "VulGuardVN":
    REPO_DIR = Path.cwd()  # local dry-run validation only

if FORCE_RECLONE and REPO_DIR.exists():
    if str(REPO_DIR).startswith("/kaggle/working/"):
        shutil.rmtree(REPO_DIR)
    else:
        raise RuntimeError("FORCE_RECLONE is only allowed under /kaggle/working")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=REPO_DIR, check=True)

CURRENT_BRANCH = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR, text=True).strip()
COMMIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Branch:", CURRENT_BRANCH)
print("Commit SHA:", COMMIT_SHA)
subprocess.run(["git", "status", "--short", "--branch"], cwd=REPO_DIR, check=True)
assert CURRENT_BRANCH == BRANCH
REVISION_DIR = REPO_DIR / "GRACE-improve" / "revision_experiments"


## E. Minimal dependencies

The cell checks imports first and installs only missing packages. It does not upgrade existing TensorFlow, PyTorch, or CUDA packages.


In [ ]:
import importlib, importlib.metadata, importlib.util, subprocess, sys

# Derived from baseline2 imports; install only packages missing from the Kaggle image.
DEPENDENCIES = {
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
    "joblib": "joblib", "matplotlib": "matplotlib", "dotenv": "python-dotenv",
    "tensorflow": "tensorflow", "torch": "torch", "transformers": "transformers",
    "accelerate": "accelerate", "bitsandbytes": "bitsandbytes", "sentencepiece": "sentencepiece",
}
requirement_files = sorted(REPO_DIR.glob("requirements*.txt"))
print("Repository requirement files:", [str(path) for path in requirement_files] or "none; using baseline2 import audit")
missing = [package for module, package in DEPENDENCIES.items() if importlib.util.find_spec(module) is None]
print("Missing packages:", missing)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-input", *missing], check=True)

for module in ("numpy", "pandas", "scipy", "sklearn", "joblib", "matplotlib", "dotenv", "tensorflow", "torch", "transformers", "accelerate", "bitsandbytes", "sentencepiece"):
    imported = importlib.import_module(module)
    package = DEPENDENCIES[module]
    try: version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: version = getattr(imported, "__version__", "unknown")
    print(f"{package}={version}")


## F. Repository smoke tests

A failing unit test stops execution before the experiment.


In [ ]:
import subprocess, sys

test_command = [sys.executable, "-m", "unittest", "discover", "-s", "GRACE-improve/revision_experiments/tests", "-p", "test_*.py", "-v"]
print(" ".join(test_command))
subprocess.run(test_command, cwd=REPO_DIR, check=True)


## G. Dry-run and execution

`RUN_MODE='dry-run'` prints and validates exactly one command, then skips execution/output packaging.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
sys.path.insert(0, str(REVISION_DIR))
from kaggle_artifacts import copy_run_artifacts, find_run, locate_or_materialize_run, next_chunk_index, restore_checkpoint

RESULTS_ROOT = Path(SMOKE_OUTPUT_ROOT if RUN_MODE == "smoke" else OUTPUT_ROOT)
RUN_DIR = RESULTS_ROOT / EXPERIMENT / DATASET / CONFIGURATION / f"seed_{SEED}"
RESTORED_CHECKPOINT = False
if RESTORE_CHECKPOINT and RUN_MODE == "full":
    restored = restore_checkpoint(CHECKPOINT_INPUT, RESULTS_ROOT, dataset=DATASET, experiment=EXPERIMENT, configuration=CONFIGURATION, seed=SEED, commit_sha=COMMIT_SHA)
    assert restored == RUN_DIR
    RESTORED_CHECKPOINT = True
    print("Restored checkpoint:", restored)
RESOLVED_CHUNK_INDEX = TEST_CHUNK_INDEX
if TEST_CHUNK_SIZE is not None and RESOLVED_CHUNK_INDEX is None:
    RESOLVED_CHUNK_INDEX = next_chunk_index(RUN_DIR, TEST_CHUNK_SIZE) if RUN_MODE == "full" else 0
print("Test chunk:", {"size": TEST_CHUNK_SIZE, "index": RESOLVED_CHUNK_INDEX})
E01_PROPOSED = find_run(RESULTS_ROOT, SOURCE_EXPERIMENT, DATASET, "proposed", SEED)
E01_BASELINE = find_run(RESULTS_ROOT, SOURCE_EXPERIMENT, DATASET, "reproduced_baseline", SEED)
if RUN_MODE != "dry-run" and REUSE_E01_RESULTS:
    if E01_PROPOSED is None:
        try: E01_PROPOSED = locate_or_materialize_run(results_root=RESULTS_ROOT, experiment=SOURCE_EXPERIMENT, dataset=DATASET, configuration="proposed", seed=SEED, input_path=E01_RESULTS_INPUT, staging_root="/kaggle/working/imported_artifacts")
        except FileNotFoundError: pass
    if E01_BASELINE is None:
        try: E01_BASELINE = locate_or_materialize_run(results_root=RESULTS_ROOT, experiment=SOURCE_EXPERIMENT, dataset=DATASET, configuration="reproduced_baseline", seed=SEED, input_path=E01_RESULTS_INPUT, staging_root="/kaggle/working/imported_artifacts")
        except FileNotFoundError: pass
REUSED_E01 = bool(not RESTORED_CHECKPOINT and CONFIGURATION == "direct_high" and REUSE_E01_RESULTS and E01_PROPOSED and RUN_MODE != "dry-run")
if REUSED_E01:
    copy_run_artifacts(E01_PROPOSED, RUN_DIR)
    config_payload = json.loads((RUN_DIR / "config.json").read_text()); config_payload.update({"experiment_name": EXPERIMENT, "configuration": CONFIGURATION, "run_directory": str(RUN_DIR), "reused_from": str(E01_PROPOSED)})
    config_payload["call_llm_for_inspect"] = True; config_payload["call_llm_for_high"] = False; config_payload["delta_high"] = 1
    (RUN_DIR / "config.json").write_text(json.dumps(config_payload, indent=2))
    metadata = json.loads((RUN_DIR / "run_metadata.json").read_text()); metadata.update({"experiment": EXPERIMENT, "configuration": CONFIGURATION, "reused_from": str(E01_PROPOSED)})
    (RUN_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2))
    print("Reused E01 proposed artifacts as direct_high:", E01_PROPOSED)
else:
    runner = REVISION_DIR / "run_revision_experiments.py"
    COMMAND = [sys.executable, str(runner), "--dataset", DATASET, "--experiment", EXPERIMENT, "--seed", str(SEED), "--configuration", CONFIGURATION, "--output-directory", str(RESULTS_ROOT), "--session-budget-hours", str(SESSION_BUDGET_HOURS), "--min-remaining-minutes", str(MIN_REMAINING_MINUTES), "--session-start-epoch", str(START_TIME.timestamp())]
    if RUN_MODE == "smoke": COMMAND += ["--smoke", "--no-resume"]
    elif RUN_MODE == "full": COMMAND += (["--resume"] if RESUME else ["--no-resume"])
    else: COMMAND += ["--dry-run"]
    if TEST_CHUNK_SIZE is not None: COMMAND += ["--test-chunk-size", str(TEST_CHUNK_SIZE), "--test-chunk-index", str(RESOLVED_CHUNK_INDEX)]
    print("Command:", " ".join(COMMAND)); os.environ["GRACE_AUTO_DOWNLOAD_MODEL"] = str(AUTO_DOWNLOAD_MODEL).lower()
    subprocess.run(COMMAND, cwd=REPO_DIR, check=True)

current_metadata = json.loads((RUN_DIR / "run_metadata.json").read_text()) if (RUN_DIR / "run_metadata.json").is_file() else {}
if RUN_MODE != "dry-run" and current_metadata.get("status") == "complete":
    routing_script = REVISION_DIR / "analyze_routing_results.py"
    routing_command = [sys.executable, str(routing_script), "--run-dir", str(RUN_DIR), "--output-path", str(RUN_DIR), "--dataset", DATASET, "--seed", str(SEED)]
    if E01_BASELINE: routing_command += ["--baseline-run", str(E01_BASELINE)]
    subprocess.run(routing_command, cwd=REPO_DIR, check=True)
elif RUN_MODE != "dry-run":
    print("Partial run: routing analysis is deferred until inference is complete.")


## H. Output validation

Required files are checked and JSON files are parsed. Missing values remain missing; they are never inferred.


In [ ]:
import json

if RUN_MODE == "dry-run":
    print("Dry-run complete; output validation is intentionally skipped.")
else:
    required = ('config.json', 'run_metadata.json', 'metrics.json', 'predictions.jsonl', 'calibration.json', 'runtime.json', 'routing_policy_metrics.json', 'routing_operating_points.csv', 'risk_coverage_curve.png', 'recall_llm_call_curve.png', 'f1_llm_call_curve.png', 'matched_recall.csv')
    missing = [name for name in required if not (RUN_DIR / name).is_file()]
    metadata = json.loads((RUN_DIR / "run_metadata.json").read_text(encoding="utf-8")) if (RUN_DIR / "run_metadata.json").is_file() else {}
    if missing and metadata.get("status") != "partial": raise FileNotFoundError(f"Missing required outputs: {missing}")
    if missing: print("Partial run; artifacts not produced yet:", missing)
    payloads = {}
    for name in required:
        if name.endswith(".json") and (RUN_DIR / name).is_file():
            payloads[name] = json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
        elif name.endswith(".jsonl") and (RUN_DIR / name).is_file():
            with (RUN_DIR / name).open("r", encoding="utf-8") as handle:
                for line_number, line in enumerate(handle, start=1):
                    if line.strip(): json.loads(line)
            print(f"Validated JSONL: {name}")
    metrics = payloads.get("metrics.json", {})
    calibration = payloads.get("calibration.json", {})
    metadata = payloads.get("run_metadata.json", metadata)
    summary = {
        "dataset": DATASET, "experiment": EXPERIMENT, "configuration": CONFIGURATION, "seed": SEED,
        "status": metadata.get("status"), "sample_count": metrics.get("samples"),
        "accuracy": metrics.get("accuracy"), "precision": metrics.get("precision"), "recall": metrics.get("recall"),
        "f1": metrics.get("f1"), "roc_auc": metrics.get("roc_auc"), "pr_auc": metrics.get("pr_auc"),
        "llm_calls": metrics.get("llm_calls"), "llm_call_ratio": metrics.get("llm_call_ratio"),
        "tau_low": calibration.get("tau_low"), "tau_high": calibration.get("tau_high"), "output_path": str(RUN_DIR),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    if metadata.get("status") not in ("complete", "partial"):
        raise RuntimeError(f"Run is neither complete nor resumable partial: {metadata.get('status')!r}")

    if metadata.get("status") == "complete":
        config_payload = payloads["config.json"]
        run_state_path = RUN_DIR / "_pipeline" / "run_state.json"
        run_signature = json.loads(run_state_path.read_text()).get("run_signature", {}) if run_state_path.is_file() else config_payload
        expected = {"direct_high": (True, False, 1), "verify_high": (True, True, 0), "reproduced_baseline": (True, True, 0)}[CONFIGURATION]
        actual = (run_signature.get("call_llm_for_inspect"), run_signature.get("call_llm_for_high"), run_signature.get("delta_high", config_payload.get("delta_high")))
        assert actual == expected, (actual, expected)
        signature_tau_low = run_signature.get("tau_low", calibration.get("tau_low"))
        signature_tau_high = run_signature.get("tau_high", calibration.get("tau_high"))
        assert signature_tau_low is not None and signature_tau_high is not None
        assert calibration.get("threshold_selection_split") == "validation", "Primary thresholds must come from validation"
        print("Run signature:", {"call_llm_for_inspect": actual[0], "call_llm_for_high": actual[1], "delta_high": actual[2], "tau_low": signature_tau_low, "tau_high": signature_tau_high})
        if CONFIGURATION == "reproduced_baseline" and RUN_MODE == "full":
            assert metrics.get("llm_calls") == metrics.get("samples") and metrics.get("llm_call_ratio") == 1.0


## I. Package results

A compact result ZIP excludes pipeline caches. If the run is partial, a separate checkpoint ZIP preserves the complete run directory, including `_pipeline`, for the next Kaggle session.


In [ ]:
if RUN_MODE == "dry-run":
    print("Dry-run: no ZIP is created.")
else:
    import sys
    sys.path.insert(0, str(REVISION_DIR))
    from kaggle_artifacts import package_checkpoint, package_run, read_json, write_run_summary
    EXPORTS_DIR = Path("/kaggle/working/exports")
    ZIP_PATH = package_run(RUN_DIR, EXPORTS_DIR, dataset=DATASET, experiment=EXPERIMENT, configuration=CONFIGURATION, seed=SEED)
    CHECKPOINT_PATH = None
    if read_json(RUN_DIR / "run_metadata.json").get("status") != "complete":
        CHECKPOINT_PATH = package_checkpoint(RUN_DIR, EXPORTS_DIR, dataset=DATASET, experiment=EXPERIMENT, configuration=CONFIGURATION, seed=SEED)
    SUMMARY_PATH = write_run_summary(EXPORTS_DIR / "run_summary.json", run_dir=RUN_DIR, commit_sha=COMMIT_SHA, seed=SEED, configuration=CONFIGURATION)
    print("ZIP:", ZIP_PATH)
    if CHECKPOINT_PATH: print("Checkpoint ZIP (upload as next session input):", CHECKPOINT_PATH)
    print("Summary:", SUMMARY_PATH)
